# NLP Assignment 3 — Transformers + RAG
**Course:** CS-4063 Natural Language Processing
**Student ID:** f230647
**System:** Encoder-only Transformer (multi-task) → Retrieval (cosine similarity) → Decoder-only Transformer (autoregressive)

Run all cells top-to-bottom. Requires PyTorch ≥ 2.0. GPU optional but recommended.

In [ ]:
import os, re, json, gzip, math, random, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from collections import Counter
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 100
from tqdm import tqdm
print("PyTorch version:", torch.__version__)
print("Device:", "cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
CONFIG = {
    'd_model': 128,
    'num_heads': 4,
    'num_encoder_layers': 3,
    'num_decoder_layers': 3,
    'd_ff': 256,
    'max_seq_len': 128,
    'max_dec_seq_len': 256,
    'vocab_size': 15000,
    'batch_size': 32,
    'encoder_epochs': 8,
    'decoder_epochs': 6,
    'lr': 3e-4,
    'dropout': 0.1,
    'top_k': 3,
    'max_gen_len': 30,
    'seed': 42,
    'data_dir': '/home/zaid/Desktop/FAST/NLP/Assignments/3/Dataset',
    'categories': ['beauty', 'cellphones', 'sports'],
    'samples_per_category': 12000,
    'models_dir': 'models',
    'results_dir': 'results',
    'alpha': 0.7,   # weight for sentiment loss vs helpfulness loss
}

torch.manual_seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
os.makedirs(CONFIG['models_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'], exist_ok=True)
print("Config loaded. Device:", DEVICE)

## Section 1: Dataset Loading & Preprocessing

We use three Amazon Review categories (Beauty, Cell Phones, Sports) to construct a balanced multi-category dataset of 36,000 samples.

In [ ]:
def load_reviews(data_dir, categories, samples_per_category, seed=42):
    """Load and sample reviews from compressed .json.gz files."""
    all_reviews = []
    rng = random.Random(seed)
    for cat in categories:
        path = os.path.join(data_dir, f"{cat}.json.gz")
        records = []
        with gzip.open(path, 'rt', encoding='utf-8') as f:
            for line in f:
                try:
                    obj = json.loads(line.strip())
                    if obj.get('reviewText') and obj.get('overall'):
                        records.append({
                            'reviewText': obj['reviewText'],
                            'overall': float(obj['overall']),
                            'summary': obj.get('summary', ''),
                            'helpful': obj.get('helpful', [0, 0]),
                            'category': cat,
                        })
                except Exception:
                    continue
        sampled = rng.sample(records, min(samples_per_category, len(records)))
        all_reviews.extend(sampled)
        print(f"  {cat}: loaded {len(sampled):,} reviews (pool: {len(records):,})")
    print(f"Total: {len(all_reviews):,} reviews")
    return all_reviews

print("Loading reviews...")
raw_data = load_reviews(CONFIG['data_dir'], CONFIG['categories'], CONFIG['samples_per_category'])

In [ ]:
def make_sentiment_label(overall):
    if overall <= 2.0:
        return 0   # Negative
    elif overall == 3.0:
        return 1   # Neutral
    else:
        return 2   # Positive

def make_helpfulness_label(helpful):
    if isinstance(helpful, list) and len(helpful) == 2:
        votes_helpful, votes_total = helpful[0], helpful[1]
        if votes_total >= 2:
            return 1 if (votes_helpful / votes_total) >= 0.5 else 0
    return 0   # default: not helpful (no evidence)

for r in raw_data:
    r['sentiment'] = make_sentiment_label(r['overall'])
    r['helpfulness'] = make_helpfulness_label(r['helpful'])

sent_counts = Counter(r['sentiment'] for r in raw_data)
help_counts = Counter(r['helpfulness'] for r in raw_data)
print("Sentiment distribution:", {0: 'Neg', 1: 'Neu', 2: 'Pos'})
for k, v in sorted(sent_counts.items()):
    print(f"  {k}: {v:,} ({100*v/len(raw_data):.1f}%)")
print("Helpfulness distribution:")
for k, v in sorted(help_counts.items()):
    print(f"  {k}: {v:,} ({100*v/len(raw_data):.1f}%)")

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'http\S+', ' ', text)
    text = re.sub(r"[^a-z0-9\s\'\-]", ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def simple_tokenize(text):
    return clean_text(text).split()

# Stratified split by sentiment label (handles 79% positive imbalance)
labels = [r['sentiment'] for r in raw_data]
indices = list(range(len(raw_data)))

sss1 = StratifiedShuffleSplit(n_splits=1, test_size=0.30, random_state=CONFIG['seed'])
train_idx, temp_idx = next(sss1.split(indices, labels))

temp_labels = [labels[i] for i in temp_idx]
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.50, random_state=CONFIG['seed'])
val_rel_idx, test_rel_idx = next(sss2.split(temp_idx, temp_labels))

val_idx = [temp_idx[i] for i in val_rel_idx]
test_idx = [temp_idx[i] for i in test_rel_idx]

train_data = [raw_data[i] for i in train_idx]
val_data   = [raw_data[i] for i in val_idx]
test_data  = [raw_data[i] for i in test_idx]

print(f"Train: {len(train_data):,} | Val: {len(val_data):,} | Test: {len(test_data):,}")

In [ ]:
class Vocabulary:
    SPECIAL = ['<PAD>', '<UNK>', '<BOS>', '<EOS>', '[SEP]']

    def __init__(self, max_size=15000):
        self.max_size = max_size
        self.word2idx = {}
        self.idx2word = {}

    def build(self, texts):
        counter = Counter()
        for text in texts:
            counter.update(simple_tokenize(text))
        # Reserve spots for specials
        vocab_words = [w for w, _ in counter.most_common(self.max_size - len(self.SPECIAL))]
        all_tokens = self.SPECIAL + vocab_words
        self.word2idx = {w: i for i, w in enumerate(all_tokens)}
        self.idx2word = {i: w for w, i in self.word2idx.items()}
        print(f"Vocabulary built: {len(self.word2idx):,} tokens")

    def encode(self, text, max_len, add_bos=False, add_eos=False):
        tokens = simple_tokenize(text)
        ids = [self.word2idx.get(t, 1) for t in tokens]  # 1 = <UNK>
        if add_bos:
            ids = [self.word2idx['<BOS>']] + ids
        if add_eos:
            ids = ids + [self.word2idx['<EOS>']]
        ids = ids[:max_len]
        ids += [0] * (max_len - len(ids))  # pad with 0 = <PAD>
        return ids

    def decode(self, indices):
        words = []
        for idx in indices:
            w = self.idx2word.get(idx, '<UNK>')
            if w == '<EOS>':
                break
            if w not in ('<PAD>', '<BOS>'):
                words.append(w)
        return ' '.join(words)

vocab = Vocabulary(max_size=CONFIG['vocab_size'])
train_texts = [r['reviewText'] + ' ' + r['summary'] for r in train_data]
vocab.build(train_texts)
print("Special tokens:", {w: vocab.word2idx[w] for w in Vocabulary.SPECIAL})

In [ ]:
class ReviewDataset(Dataset):
    def __init__(self, data, vocab, max_seq_len):
        self.data = data
        self.vocab = vocab
        self.max_seq_len = max_seq_len

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        r = self.data[idx]
        input_ids = torch.tensor(
            self.vocab.encode(r['reviewText'], self.max_seq_len),
            dtype=torch.long
        )
        return {
            'input_ids': input_ids,
            'sentiment': torch.tensor(r['sentiment'], dtype=torch.long),
            'helpfulness': torch.tensor(r['helpfulness'], dtype=torch.long),
            'idx': idx,
        }

train_ds = ReviewDataset(train_data, vocab, CONFIG['max_seq_len'])
val_ds   = ReviewDataset(val_data,   vocab, CONFIG['max_seq_len'])
test_ds  = ReviewDataset(test_data,  vocab, CONFIG['max_seq_len'])

train_loader = DataLoader(train_ds, batch_size=CONFIG['batch_size'], shuffle=True,  num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=CONFIG['batch_size'], shuffle=False, num_workers=0)

print(f"Loaders created. Train batches: {len(train_loader)}")

In [ ]:
# Sanity checks
fig, axes = plt.subplots(1, 2, figsize=(10, 3))
sent_labels = ['Negative', 'Neutral', 'Positive']
help_labels = ['Not Helpful', 'Helpful']
axes[0].bar(sent_labels, [sent_counts[i] for i in range(3)], color=['#e74c3c', '#f39c12', '#2ecc71'])
axes[0].set_title('Sentiment Distribution (full dataset)')
axes[0].set_ylabel('Count')
axes[1].bar(help_labels, [help_counts[i] for i in range(2)], color=['#e74c3c', '#2ecc71'])
axes[1].set_title('Helpfulness Distribution (full dataset)')
axes[1].set_ylabel('Count')
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'class_distribution.png'), bbox_inches='tight')
plt.show()

# Sample review
sample = train_data[0]
print("Sample review:", sample['reviewText'][:150])
print("Rating:", sample['overall'], "-> Sentiment:", sent_labels[sample['sentiment']])
print("Helpful:", sample['helpful'], "-> Helpfulness:", sample['helpfulness'])
# Vocab coverage check
unk_count = sum(
    1 for t in simple_tokenize(train_data[0]['reviewText'])
    if vocab.word2idx.get(t, 1) == 1
)
print(f"UNK tokens in sample review: {unk_count}/{len(simple_tokenize(train_data[0]['reviewText']))}")

## Section 2: Part A — Encoder Architecture (from scratch)

All transformer components are implemented manually — no `nn.Transformer`, `nn.MultiheadAttention`, or `nn.TransformerEncoder`.

**Design choices:**
- **Pre-LN (LayerNorm before sublayer):** More stable gradient flow vs. Post-LN, no warmup needed
- **Additive masking:** -∞ mask composes naturally; `causal + padding` via simple addition
- **Mean pooling:** More robust than CLS-token for sentence embeddings on short sequences

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_seq_len, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_seq_len, d_model)
        pos = torch.arange(max_seq_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))  # (1, T, D)

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])

In [ ]:
def scaled_dot_product_attention(Q, K, V, mask=None):
    """
    Q, K, V: (B, H, T, d_k)
    mask:    additive mask — 0 for attended, -inf for masked positions
    Returns: output (B, H, T, d_v), weights (B, H, T, T_k)
    """
    d_k = Q.size(-1)
    scores = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(d_k)
    if mask is not None:
        scores = scores + mask
    attn = F.softmax(scores, dim=-1)
    return torch.matmul(attn, V), attn


class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads, dropout=0.1):
        super().__init__()
        assert d_model % num_heads == 0
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads
        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model)
        self.dropout = nn.Dropout(dropout)

    def split_heads(self, x):
        B, T, _ = x.shape
        return x.view(B, T, self.num_heads, self.d_k).transpose(1, 2)

    def forward(self, query, key, value, mask=None):
        B = query.size(0)
        Q = self.split_heads(self.W_Q(query))
        K = self.split_heads(self.W_K(key))
        V = self.split_heads(self.W_V(value))
        out, attn = scaled_dot_product_attention(Q, K, V, mask)
        out = out.transpose(1, 2).contiguous().view(B, -1, self.d_model)
        return self.dropout(self.W_O(out)), attn

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model),
        )

    def forward(self, x):
        return self.net(x)


class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super().__init__()
        self.self_attn = MultiHeadAttention(d_model, num_heads, dropout)
        self.ff = FeedForward(d_model, d_ff, dropout)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, src_mask=None):
        # Pre-LN: normalize before sublayer, then residual
        normed = self.norm1(x)
        attn_out, _ = self.self_attn(normed, normed, normed, src_mask)
        x = x + self.dropout(attn_out)
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

In [ ]:
class ReviewEncoder(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, d_ff,
                 max_seq_len, num_sentiment=3, dropout=0.1):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model, padding_idx=0)
        self.pos_enc = PositionalEncoding(d_model, max_seq_len, dropout)
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)
        self.sentiment_head = nn.Linear(d_model, num_sentiment)
        self.helpfulness_head = nn.Linear(d_model, 2)

    def make_padding_mask(self, src):
        # (B, T) -> (B, 1, 1, T): -inf where pad, 0 elsewhere
        mask = (src == 0).float() * -1e9
        return mask.unsqueeze(1).unsqueeze(2)

    def forward(self, src):
        mask = self.make_padding_mask(src)
        x = self.pos_enc(self.embedding(src) * math.sqrt(self.d_model))
        for layer in self.layers:
            x = layer(x, mask)
        x = self.norm(x)
        # Mean pooling over non-padding positions
        pad_mask = (src != 0).float().unsqueeze(-1)
        embeddings = (x * pad_mask).sum(1) / pad_mask.sum(1).clamp(min=1)
        return self.sentiment_head(embeddings), self.helpfulness_head(embeddings), embeddings


encoder = ReviewEncoder(
    vocab_size=len(vocab.word2idx),
    d_model=CONFIG['d_model'],
    num_heads=CONFIG['num_heads'],
    num_layers=CONFIG['num_encoder_layers'],
    d_ff=CONFIG['d_ff'],
    max_seq_len=CONFIG['max_seq_len'],
    dropout=CONFIG['dropout'],
).to(DEVICE)

n_params = sum(p.numel() for p in encoder.parameters())
print(f"Encoder parameters: {n_params:,}")

## Section 3: Part A — Multi-Task Training

**Loss:** `L = 0.7 × L_sentiment + 0.3 × L_helpfulness` (weighted CE for both to handle class imbalance)
**Optimizer:** AdamW with CosineAnnealingLR and gradient clipping (max_norm=1.0)

In [ ]:
def compute_class_weights(data, key, num_classes):
    counts = Counter(r[key] for r in data)
    total = len(data)
    weights = [total / (num_classes * counts.get(i, 1)) for i in range(num_classes)]
    return torch.tensor(weights, dtype=torch.float32).to(DEVICE)

sent_weights = compute_class_weights(train_data, 'sentiment', 3)
help_weights = compute_class_weights(train_data, 'helpfulness', 2)
print("Sentiment class weights:", sent_weights.tolist())
print("Helpfulness class weights:", help_weights.tolist())

sent_criterion = nn.CrossEntropyLoss(weight=sent_weights)
help_criterion = nn.CrossEntropyLoss(weight=help_weights)

In [ ]:
def evaluate_encoder(model, loader):
    model.eval()
    all_sent_pred, all_sent_true = [], []
    all_help_pred, all_help_true = [], []
    total_loss = 0.0
    with torch.no_grad():
        for batch in loader:
            src = batch['input_ids'].to(DEVICE)
            s_true = batch['sentiment'].to(DEVICE)
            h_true = batch['helpfulness'].to(DEVICE)
            s_logits, h_logits, _ = model(src)
            loss = CONFIG['alpha'] * sent_criterion(s_logits, s_true) +                    (1 - CONFIG['alpha']) * help_criterion(h_logits, h_true)
            total_loss += loss.item()
            all_sent_pred.extend(s_logits.argmax(1).cpu().tolist())
            all_sent_true.extend(s_true.cpu().tolist())
            all_help_pred.extend(h_logits.argmax(1).cpu().tolist())
            all_help_true.extend(h_true.cpu().tolist())
    avg_loss = total_loss / len(loader)
    sent_acc = sum(p == t for p, t in zip(all_sent_pred, all_sent_true)) / len(all_sent_true)
    help_acc = sum(p == t for p, t in zip(all_help_pred, all_help_true)) / len(all_help_true)
    return avg_loss, sent_acc, help_acc, all_sent_pred, all_sent_true, all_help_pred, all_help_true


def train_encoder(model, train_loader, val_loader):
    optimizer = torch.optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CONFIG['encoder_epochs'])
    history = {'train_loss': [], 'val_loss': [], 'train_sent_acc': [], 'val_sent_acc': [],
               'train_help_acc': [], 'val_help_acc': []}

    for epoch in range(CONFIG['encoder_epochs']):
        model.train()
        total_loss, total_sent_correct, total_help_correct, total = 0, 0, 0, 0
        for batch in tqdm(train_loader, desc=f'Epoch {epoch+1}/{CONFIG["encoder_epochs"]}', leave=False):
            src = batch['input_ids'].to(DEVICE)
            s_true = batch['sentiment'].to(DEVICE)
            h_true = batch['helpfulness'].to(DEVICE)
            optimizer.zero_grad()
            s_logits, h_logits, _ = model(src)
            loss = CONFIG['alpha'] * sent_criterion(s_logits, s_true) +                    (1 - CONFIG['alpha']) * help_criterion(h_logits, h_true)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()
            total_sent_correct += (s_logits.argmax(1) == s_true).sum().item()
            total_help_correct += (h_logits.argmax(1) == h_true).sum().item()
            total += src.size(0)
        scheduler.step()

        train_sent_acc = total_sent_correct / total
        train_help_acc = total_help_correct / total
        train_loss = total_loss / len(train_loader)
        val_loss, val_sent_acc, val_help_acc, *_ = evaluate_encoder(model, val_loader)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_sent_acc'].append(train_sent_acc)
        history['val_sent_acc'].append(val_sent_acc)
        history['train_help_acc'].append(train_help_acc)
        history['val_help_acc'].append(val_help_acc)

        print(f"Epoch {epoch+1:2d} | Loss {train_loss:.4f}/{val_loss:.4f} | "
              f"Sent Acc {train_sent_acc:.3f}/{val_sent_acc:.3f} | "
              f"Help Acc {train_help_acc:.3f}/{val_help_acc:.3f}")
    return history

print("Starting encoder training...")
enc_history = train_encoder(encoder, train_loader, val_loader)
print("Encoder training complete!")

In [ ]:
# Learning curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
epochs = range(1, CONFIG['encoder_epochs'] + 1)
axes[0].plot(epochs, enc_history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs, enc_history['val_loss'], label='Val', marker='s')
axes[0].set_title('Combined Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend()
axes[1].plot(epochs, enc_history['train_sent_acc'], label='Train', marker='o')
axes[1].plot(epochs, enc_history['val_sent_acc'], label='Val', marker='s')
axes[1].set_title('Sentiment Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].legend()
axes[2].plot(epochs, enc_history['train_help_acc'], label='Train', marker='o')
axes[2].plot(epochs, enc_history['val_help_acc'], label='Val', marker='s')
axes[2].set_title('Helpfulness Accuracy'); axes[2].set_xlabel('Epoch'); axes[2].legend()
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'encoder_curves.png'), bbox_inches='tight')
plt.show()

In [ ]:
# Test set evaluation
_, _, _, sent_pred, sent_true, help_pred, help_true = evaluate_encoder(encoder, test_loader)
print("=== Sentiment Classification Report ===")
print(classification_report(sent_true, sent_pred, target_names=['Negative', 'Neutral', 'Positive']))
print("=== Helpfulness Classification Report ===")
print(classification_report(help_true, help_pred, target_names=['Not Helpful', 'Helpful']))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, cm_data, cm_labels, title in [
    (axes[0], confusion_matrix(sent_true, sent_pred), ['Neg', 'Neu', 'Pos'], 'Sentiment Confusion Matrix'),
    (axes[1], confusion_matrix(help_true, help_pred), ['Not Help', 'Helpful'], 'Helpfulness Confusion Matrix'),
]:
    cm_norm = cm_data.astype(float) / cm_data.sum(axis=1, keepdims=True)
    ax.imshow(cm_norm, cmap='Blues')
    ax.set_xticks(range(len(cm_labels))); ax.set_yticks(range(len(cm_labels)))
    ax.set_xticklabels(cm_labels); ax.set_yticklabels(cm_labels)
    ax.set_xlabel('Predicted'); ax.set_ylabel('True'); ax.set_title(title)
    for i in range(cm_norm.shape[0]):
        for j in range(cm_norm.shape[1]):
            ax.text(j, i, f'{cm_norm[i,j]:.2f}', ha='center', va='center', fontsize=9)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['results_dir'], 'confusion_matrices.png'), bbox_inches='tight')
plt.show()

## Section 4: Embedding Extraction

Extract 128-dim review embeddings from the encoder for all training samples. These are used by the retrieval module.

In [ ]:
def extract_embeddings(model, loader, data):
    model.eval()
    all_embs, all_meta = [], []
    with torch.no_grad():
        for batch in tqdm(loader, desc='Extracting embeddings'):
            src = batch['input_ids'].to(DEVICE)
            s_logits, h_logits, embs = model(src)
            all_embs.append(embs.cpu().numpy())
            s_pred = s_logits.argmax(1).cpu().tolist()
            h_pred = h_logits.argmax(1).cpu().tolist()
            for i, orig_idx in enumerate(batch['idx'].tolist()):
                all_meta.append({
                    'orig_idx': orig_idx,
                    'sent_pred': s_pred[i],
                    'help_pred': h_pred[i],
                    'sent_true': data[orig_idx]['sentiment'],
                    'help_true': data[orig_idx]['helpfulness'],
                })
    return np.concatenate(all_embs, axis=0), all_meta

train_embeddings, train_meta = extract_embeddings(encoder, train_loader, train_data)
np.save(os.path.join(CONFIG['results_dir'], 'train_embeddings.npy'), train_embeddings)
with open(os.path.join(CONFIG['results_dir'], 'train_metadata.json'), 'w') as f:
    json.dump(train_meta, f)
torch.save(encoder.state_dict(), os.path.join(CONFIG['models_dir'], 'encoder.pt'))
print(f"Saved embeddings: {train_embeddings.shape}")
print(f"Saved encoder weights: models/encoder.pt")